# Prompting Engine

## Objective

This engine converts retrieved knowledge into a complete prompt ready for inference.

Inputs

• metadata/retrievals.json

• retrieval/*.json

Outputs

• prompts/*.json

• metadata/prompts.json

The Generation Engine consumes these prompt objects directly.

In [1]:
!pip install -q transformers sentence-transformers


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
import json

from pathlib import Path

In [6]:
ROOT = Path("/content/drive/MyDrive/MicroBrain")

DATA = ROOT / "Data"

RAW = DATA / "raw"

PROCESSED = DATA / "processed"

CHUNKS = DATA / "chunks"

EMBEDDINGS = DATA / "embeddings"

RETRIEVAL = DATA / "retrieval"

PROMPTS = DATA / "prompts"

VECTORDB = DATA / "vectordb"

METADATA = DATA / "metadata"

for folder in [
    RAW,
    PROCESSED,
    CHUNKS,
    EMBEDDINGS,
    RETRIEVAL,
    PROMPTS,
    VECTORDB,
    METADATA
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

In [7]:
print(PROMPTS)

print()

print(METADATA)

/content/drive/MyDrive/MicroBrain/Data/prompts

/content/drive/MyDrive/MicroBrain/Data/metadata


## Step 7 – Load Retrieval Registry

Load the retrieval registry produced by the Retrieval Engine.

This registry tells us which retrieval objects are available for prompt construction.

In [8]:
registry_path = METADATA / "retrievals.json"

with open(registry_path, "r", encoding="utf-8") as file:
    retrieval_registry = json.load(file)

print(type(retrieval_registry))
print()

print(len(retrieval_registry))


<class 'list'>

1


## Step 8 – Inspect Retrieval Registry

Verify that the registry contains the expected retrieval metadata.

In [9]:
print()

print(retrieval_registry[0])


{'retrieval_id': 'e0348682-db79-4bc2-a545-b6eb73723a1a', 'question': 'How do transformers convert words into vectors?', 'top_k': 3, 'retrieval_file': 'e0348682-db79-4bc2-a545-b6eb73723a1a.json'}


## Step 9 – Load Retrieval Object

Load the complete retrieval object from disk.

This contains the user question and the retrieved context that will be used to build the prompt.

In [10]:
retrieval_info = retrieval_registry[0]

retrieval_path = RETRIEVAL / retrieval_info["retrieval_file"]

with open(retrieval_path, "r", encoding="utf-8") as file:
    retrieval_object = json.load(file)

print(type(retrieval_object))

<class 'dict'>


## Step 10 – Validate Retrieval Object

Verify that the Prompting Engine received the complete retrieval object.

In [11]:
print()

print(retrieval_object.keys())


dict_keys(['retrieval_id', 'question', 'top_k', 'retrieved_chunks', 'context'])


## Step 11 – Define the System Prompt

Create the instruction that guides the language model's behaviour.

The same system prompt can later be swapped for different applications without modifying the retrieval or generation engines.

In [12]:
SYSTEM_PROMPT = """
You are a helpful AI assistant.

Answer the user's question using ONLY the retrieved context provided.

If the answer cannot be found in the retrieved context, clearly state that the information is unavailable.

Be concise, factual and accurate.
""".strip()

print(SYSTEM_PROMPT)

You are a helpful AI assistant.

Answer the user's question using ONLY the retrieved context provided.

If the answer cannot be found in the retrieved context, clearly state that the information is unavailable.

Be concise, factual and accurate.


## Step 12 – Extract User Question

Retrieve the original user question from the retrieval object.

In [13]:
question = retrieval_object["question"]

print(question)

How do transformers convert words into vectors?


## Step 13 – Assemble Retrieved Context

Combine the retrieved chunk texts into a single context block.

The Retrieval Engine already extracts the text from each chunk, so the Prompting Engine only needs to concatenate them.

In [17]:
context = "\n\n".join(

    chunk["text"]

    for chunk in retrieval_object["retrieved_chunks"]

)

print(type(context))

print()

print(len(context))

<class 'str'>

1504


In [18]:
print("=" * 60)

print(context[:1000])

print("=" * 60)


learned computation involved. Step 2 — Integers Become Vectors (Embeddings) In plain terms A single number can't describe what a word means — so instead, each token ID is turned into a long list of numbers (a vector). Imagine coordinates on a map, except instead of just latitude and longitude, there are hundreds of coordinates, each capturing a tiny shade of meaning: how "animal-like", how "large", how "emotional" a concept is. No single number matters on its own — together, they define meaning.

ddings — Reusing the Same Math In plain terms Just like individual words get turned into meaning-vectors, entire documents (or chunks of them) get their own meaning-vectors too. A question gets converted the same way. Then the system simply checks which document-vectors sit closest to the question-vector — the closer they are, the more relevant that document probably is. The math similarity(q, d) = (q · d) / (||q|| × ||d||) Technical detail Both documents and the incoming query are passed thro

## Step 15 – Construct Prompt

Combine the system instructions, retrieved context and user question into a single prompt.

The Generation Engine will send this prompt directly to the language model.

In [19]:
prompt = f"""
System

{SYSTEM_PROMPT}

----------------------------------------

Retrieved Context

{context}

----------------------------------------

User Question

{question}

Assistant
""".strip()

print(prompt[:1200])


System

You are a helpful AI assistant.

Answer the user's question using ONLY the retrieved context provided.

If the answer cannot be found in the retrieved context, clearly state that the information is unavailable.

Be concise, factual and accurate.

----------------------------------------

Retrieved Context

learned computation involved. Step 2 — Integers Become Vectors (Embeddings) In plain terms A single number can't describe what a word means — so instead, each token ID is turned into a long list of numbers (a vector). Imagine coordinates on a map, except instead of just latitude and longitude, there are hundreds of coordinates, each capturing a tiny shade of meaning: how "animal-like", how "large", how "emotional" a concept is. No single number matters on its own — together, they define meaning.

ddings — Reusing the Same Math In plain terms Just like individual words get turned into meaning-vectors, entire documents (or chunks of them) get their own meaning-vectors too. A qu

## Step 16 – Create Prompt Object

Create the standardized prompt object that will be consumed by the Generation Engine.

In [20]:
import uuid

prompt_object = {

    "prompt_id": str(uuid.uuid4()),

    "retrieval_id": retrieval_object["retrieval_id"],

    "question": question,

    "system_prompt": SYSTEM_PROMPT,

    "context": context,

    "prompt": prompt

}

print(prompt_object.keys())

dict_keys(['prompt_id', 'retrieval_id', 'question', 'system_prompt', 'context', 'prompt'])


## Step 17 – Validate Prompt Object

Inspect the prompt object before saving it to disk.

In [21]:
print()

print(prompt_object["prompt_id"])

print()

print(prompt_object["question"])

print()

print(prompt_object["prompt"][:1000])


ab43c215-1201-4a34-83c5-d55e47821c94

How do transformers convert words into vectors?

System

You are a helpful AI assistant.

Answer the user's question using ONLY the retrieved context provided.

If the answer cannot be found in the retrieved context, clearly state that the information is unavailable.

Be concise, factual and accurate.

----------------------------------------

Retrieved Context

learned computation involved. Step 2 — Integers Become Vectors (Embeddings) In plain terms A single number can't describe what a word means — so instead, each token ID is turned into a long list of numbers (a vector). Imagine coordinates on a map, except instead of just latitude and longitude, there are hundreds of coordinates, each capturing a tiny shade of meaning: how "animal-like", how "large", how "emotional" a concept is. No single number matters on its own — together, they define meaning.

ddings — Reusing the Same Math In plain terms Just like individual words get turned into meani

## Step 18 – Save Prompt Object

Persist the prompt object to disk so it can be consumed by the Generation Engine.

Each prompt is stored as an individual JSON document.

In [22]:
prompt_file = PROMPTS / f"{prompt_object['prompt_id']}.json"

with open(
    prompt_file,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        prompt_object,
        file,
        indent=4,
        ensure_ascii=False
    )

print(prompt_file)

/content/drive/MyDrive/MicroBrain/Data/prompts/ab43c215-1201-4a34-83c5-d55e47821c94.json


## Step 19 – Update Prompt Registry

Maintain a registry of every prompt generated by the Prompting Engine.

In [23]:
registry_path = METADATA / "prompts.json"

if registry_path.exists():

    with open(
        registry_path,
        "r",
        encoding="utf-8"
    ) as file:

        registry = json.load(file)

else:

    registry = []

In [24]:
registry.append({

    "prompt_id": prompt_object["prompt_id"],

    "retrieval_id": prompt_object["retrieval_id"],

    "question": question,

    "prompt_file": prompt_file.name

})

In [25]:
with open(
    registry_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        registry,
        file,
        indent=4
    )

print(registry_path)

/content/drive/MyDrive/MicroBrain/Data/metadata/prompts.json


## Step 20 – Validate Prompt Registry

Verify that the prompt registry contains the expected prompt metadata.

In [26]:
with open(
    METADATA / "prompts.json",
    "r",
    encoding="utf-8"
) as file:

    registry = json.load(file)

print(type(registry))
print(len(registry))

print()

print(registry[0])


<class 'list'>
1

{'prompt_id': 'ab43c215-1201-4a34-83c5-d55e47821c94', 'retrieval_id': 'e0348682-db79-4bc2-a545-b6eb73723a1a', 'question': 'How do transformers convert words into vectors?', 'prompt_file': 'ab43c215-1201-4a34-83c5-d55e47821c94.json'}


## Step 21 – Final Validation

Verify that the Prompting Engine produced all expected outputs.

Outputs:

• Prompt Object

• Prompt Registry

The Generation Engine will consume these outputs directly.

In [27]:
print("Prompt Objects")
print()

for file in PROMPTS.glob("*.json"):
    print(file.name)

print()

print("Metadata")
print()

for file in METADATA.glob("*prompt*"):
    print(file.name)

Prompt Objects

ab43c215-1201-4a34-83c5-d55e47821c94.json

Metadata

prompts.json
